# Trois contrôles de l'étape 3 tirés de la revue de littérature

Décisions écrites avant calcul à la section VI ter de `research/plan_projet.md`. Ce carnet ne modifie aucune sortie de l'étape 3 ; il écrit les siennes sous le préfixe `consolidation_`.

In [1]:
import sys, json
from math import erfc, sqrt
from pathlib import Path

import numpy as np
import pandas as pd

RACINE = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / ".git").exists())
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))
from src import risque, regles
from src.empreintes import empreinte

SORTIE = RACINE / "data" / "processed"
tout = pd.read_csv(SORTIE / "valeurs_portefeuilles.csv", index_col=0).join(
    pd.read_csv(SORTIE / "valeurs_comparaisons.csv", index_col=0))
LAMBDA, FENETRE = 0.94, 252

def p_chi2_1(lr):
    return erfc(sqrt(lr / 2)) if lr == lr else float("nan")
print(tout.shape)

(6709, 25)


## 1. VaR historique et VaR ajustée de la volatilité, éprouvées par Kupiec et Christoffersen

La VaR historique reprend exactement la règle de l'étape 3 : quantile des 252 séances précédentes. La VaR ajustée divise chaque rendement passé par la volatilité EWMA connue la veille de ce rendement, lit le quantile, puis le multiplie par la volatilité EWMA prévue pour le lendemain.

In [2]:
lignes = []
for serie in tout.columns:
    r = tout[serie].dropna().pct_change().dropna()
    if len(r) < FENETRE + 250:
        continue
    variance = (r ** 2).ewm(alpha=1 - LAMBDA, adjust=False).mean().shift(1)   # connue la veille
    sigma = np.sqrt(variance)
    z = r / sigma
    for seuil in (0.99, 0.95):
        historique = r.rolling(FENETRE).quantile(1 - seuil).shift(1)
        ajustee = z.rolling(FENETRE).quantile(1 - seuil).shift(1) * sigma
        for methode, var in [("historique", historique), ("ajustee_volatilite", ajustee)]:
            d = pd.concat([r.rename("r"), var.rename("v")], axis=1).dropna()
            d = d.iloc[FENETRE:] if methode == "ajustee_volatilite" else d
            depasse = (d.r < d.v).astype(int)
            n, x = len(d), int(depasse.sum())
            k = risque.kupiec(n, x, 1 - seuil)
            c = regles.christoffersen(depasse.to_numpy())
            lignes.append({"serie": serie, "seuil": seuil, "methode": methode, "obs": n, "depassements": x,
                           "taux": x / n, "kupiec": k, "p_kupiec": p_chi2_1(k),
                           "christoffersen": c, "p_christoffersen": p_chi2_1(c)})
var_controle = pd.DataFrame(lignes)
var_controle.to_csv(SORTIE / "consolidation_var.csv", index=False, encoding="utf-8", lineterminator="\n")
for seuil in (0.99, 0.95):
    bloc = var_controle[var_controle.seuil == seuil]
    print(f"=== VaR {seuil:.0%}")
    print(bloc.groupby("methode").agg(series=("serie", "size"), taux_median=("taux", "median"),
          rejets_kupiec=("p_kupiec", lambda p: int((p < 0.05).sum())),
          rejets_christoffersen=("p_christoffersen", lambda p: int((p < 0.05).sum()))).round(4).to_string())
    print()
print(var_controle[var_controle.serie.isin(["P1_reeq", "P1_cons", "SPY"]) & (var_controle.seuil == 0.99)]
      .round(4).to_string(index=False))

=== VaR 99%
                    series  taux_median  rejets_kupiec  rejets_christoffersen
methode                                                                      
ajustee_volatilite      25       0.0139             23                     12
historique              25       0.0164             25                     23

=== VaR 95%
                    series  taux_median  rejets_kupiec  rejets_christoffersen
methode                                                                      
ajustee_volatilite      25       0.0529              1                     11
historique              25       0.0556             13                     25

  serie  seuil            methode  obs  depassements   taux  kupiec  p_kupiec  christoffersen  p_christoffersen
P1_reeq   0.99         historique 6456           108 0.0167 24.5557    0.0000         23.0045            0.0000
P1_reeq   0.99 ajustee_volatilite 6203            86 0.0139  8.3511    0.0039          1.9984            0.1575
P1_cons   0.99

## 2. La part du risque de `P5` et `P9` sous la covariance de Ledoit et Wolf

Même méthode qu'à la tâche 19 de l'étape 3 : poids du dernier relevé mensuel, 252 dernières séances. Les deux covariances sont calculées sur le même bloc, les séances absentes valant zéro ; la covariance par paires de l'étape 3 est rappelée à côté.

In [3]:
rendements = pd.read_csv(SORTIE / "rendements_valorisation.csv", index_col=0)
poids_mensuels = pd.read_csv(SORTIE / "poids_mensuels.csv")
appartenance = pd.read_csv(SORTIE / "appartenance.csv")
derniere = poids_mensuels.date.max()
NOMS = {"P4": "acheteurs", "P5": "vendeurs a la chaine", "P6": "electricite", "P7": "equipement",
        "P8": "immobilier", "P9": "fournisseurs technologiques", "P10": "autres fournisseurs"}
groupe = {t: n for pf, n in NOMS.items() for t in appartenance[appartenance.portefeuille == pf].titre}

lignes = []
for serie in ["P1_reeq", "P1_cons"]:
    w = poids_mensuels[(poids_mensuels.serie == serie) & (poids_mensuels.date == derniere)
                       & (poids_mensuels.titre != "_tresorerie")].set_index("titre").poids
    w = w[(w > 0) & w.index.isin(rendements.columns)]
    w = w / w.sum()
    bloc = rendements[list(w.index)].iloc[-252:]
    lw, intensite = regles.ledoit_wolf(bloc.fillna(0).to_numpy())
    covs = {"paires_etape3": bloc.cov().to_numpy(), "empirique_meme_bloc": np.cov(bloc.fillna(0).to_numpy(), rowvar=False, ddof=0),
            "ledoit_wolf": lw}
    for nom, cov in covs.items():
        contrib, vol = risque.contributions_au_risque(w.to_numpy(), cov)
        part = pd.Series(contrib / vol, index=w.index).groupby(w.index.map(groupe)).sum()
        lignes.append({"serie": serie, "covariance": nom, "volatilite": vol * np.sqrt(252),
                       "P5": part["vendeurs a la chaine"], "P9": part["fournisseurs technologiques"],
                       "P5_plus_P9": part["vendeurs a la chaine"] + part["fournisseurs technologiques"],
                       "electricite": part["electricite"], "retrecissement": intensite if nom == "ledoit_wolf" else np.nan})
parts_lw = pd.DataFrame(lignes)
parts_lw.to_csv(SORTIE / "consolidation_parts_ledoit_wolf.csv", index=False, encoding="utf-8", lineterminator="\n")
print("au", derniere)
print(parts_lw.round(4).to_string(index=False))

au 2026-09-04
  serie          covariance  volatilite     P5     P9  P5_plus_P9  electricite  retrecissement
P1_reeq       paires_etape3      0.2493 0.6182 0.1480      0.7662       0.0204             NaN
P1_reeq empirique_meme_bloc      0.2482 0.6196 0.1461      0.7657       0.0205             NaN
P1_reeq         ledoit_wolf      0.2431 0.6195 0.1460      0.7655       0.0206           0.042
P1_cons       paires_etape3      0.4099 0.8143 0.0288      0.8431       0.0120             NaN
P1_cons empirique_meme_bloc      0.4086 0.8151 0.0279      0.8430       0.0120             NaN
P1_cons         ledoit_wolf      0.4009 0.8156 0.0277      0.8434       0.0119           0.042


## 3. La volatilité annualisée corrigée des autocorrélations

Variance de long terme à noyau de Bartlett sur 21 retards, comparée aux annualisations quotidienne et mensuelle de l'étape 3.

In [4]:
def variance_long_terme(r, retards=21):
    x = r - r.mean()
    gamma0 = (x ** 2).mean()
    total = gamma0
    for k in range(1, retards + 1):
        total += 2 * (1 - k / (retards + 1)) * (x[k:].to_numpy() * x[:-k].to_numpy()).mean()
    return total

lignes = []
for serie in tout.columns:
    niveau = tout[serie].dropna()
    r = niveau.pct_change().dropna()
    mensuel = niveau.groupby(pd.Series(niveau.index).str[:7].values).last().pct_change().dropna()
    lignes.append({"serie": serie, "quotidienne": risque.volatilite(r),
                   "corrigee_21_retards": float(np.sqrt(252 * variance_long_terme(r))),
                   "mensuelle": risque.volatilite(mensuel, risque.MOIS)})
annualisation = pd.DataFrame(lignes).set_index("serie")
annualisation["ecart_quotidien_mensuel"] = annualisation.quotidienne / annualisation.mensuelle - 1
annualisation["ecart_corrige_mensuel"] = annualisation.corrigee_21_retards / annualisation.mensuelle - 1
annualisation.to_csv(SORTIE / "consolidation_annualisation.csv", encoding="utf-8", lineterminator="\n")
print(annualisation.round(4).to_string())
print()
print("ecart median a la mensuelle : quotidienne %+.1f %%, corrigee %+.1f %%"
      % (100 * annualisation.ecart_quotidien_mensuel.median(), 100 * annualisation.ecart_corrige_mensuel.median()))
print("series ou la correction rapproche de la mensuelle : %d sur %d"
      % ((annualisation.ecart_corrige_mensuel.abs() < annualisation.ecart_quotidien_mensuel.abs()).sum(), len(annualisation)))

          quotidienne  corrigee_21_retards  mensuelle  ecart_quotidien_mensuel  ecart_corrige_mensuel
serie                                                                                                
P1_reeq        0.2173               0.1981     0.1910                   0.1375                 0.0371
P1_cons        0.2352               0.2100     0.1997                   0.1777                 0.0516
P10_reeq       0.2302               0.2113     0.1986                   0.1590                 0.0638
P10_cons       0.2722               0.2643     0.2592                   0.0504                 0.0201
P2_reeq        0.2382               0.2194     0.2146                   0.1097                 0.0224
P2_cons        0.2556               0.2303     0.2212                   0.1552                 0.0411
P3_reeq        0.1846               0.1645     0.1517                   0.2166                 0.0838
P3_cons        0.2300               0.2205     0.2164                   0.0632    

In [5]:
sorties = ["consolidation_var.csv", "consolidation_parts_ledoit_wolf.csv", "consolidation_annualisation.csv"]
(SORTIE / "consolidation_manifest.json").write_text(json.dumps({
    "produit_par": "src/consolider_etape3.ipynb", "decisions": "research/plan_projet.md, section VI ter",
    "code_sha256": {"src/regles.py": empreinte(RACINE / "src" / "regles.py"), "src/risque.py": empreinte(RACINE / "src" / "risque.py")},
    "sorties_sha256": {s: empreinte(SORTIE / s) for s in sorties}}, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("ecrit :", sorties)

ecrit : ['consolidation_var.csv', 'consolidation_parts_ledoit_wolf.csv', 'consolidation_annualisation.csv']
